In [ ]:
import pandas as pd
import numpy as np
from ehull_utils import (
    collect_mlip_energies_to_df,
    construct_phase_diagrams,
    summarize_results,
)
import os
from pathlib import Path
import glob
from pymatgen.core import Structure

In [ ]:
hdp_df = pd.read_csv(
    "HDP_CombinedInfo_WithChemSys.csv",
    index_col=0,
)
hdp_df.set_index("comp", inplace=True)
subsys_df = pd.read_csv(
    "hdp_mp_subsysandhdps.csv",
    index_col=0,
)
subsys_df["structure"] = subsys_df.apply(
    lambda row: Structure.from_dict(eval(row["structure_dict"])), axis=1
)
# mlip_df = pd.read_csv("/home/lwalterb/hdp_project/HDP_WorkFlow_Analysis/E_above_hull/hdp_mlipenergies_full.csv",index_col=0)
# mlip_df['structure'] = mlip_df.apply(lambda row: Structure.from_dict(eval(row['structure_dict'])),axis=1)

In [ ]:
data_dirs = ["MLIP_Results"]
for data_dir in data_dirs:
    # mlip_list = [x.split("_")[-1].strip('.json') for x in glob.glob(f'{data_dir}/*.json')]
    mlip_list = [
        "GRACE-3L-OMAT-large-ft-AM",
        "SevenNet-MPALOE",
        "SevenNet-omat24",
        "PET-OAM-XL",
    ]  # ,   ['Equiformer-v3']
    print(mlip_list)
    print(f"collecting data from {data_dir}")
    mlip_df = collect_mlip_energies_to_df(
        structures_df=subsys_df,
        mlip_list=mlip_list,
        data_dir=data_dir,
        output_fn=os.path.join(data_dir, f"hdp_mlipenergies_{len(mlip_list)}MLIPS.csv"),
    )
    # Failed convergence strings and NaNs need to be replaces with None to assure functionality
    mask = mlip_df.astype(str).apply(lambda col: col.str.contains("Failed", na=True))
    mlip_df = mlip_df.mask(mask, None)
    # print(mlip_df.head())
    print(f"mlipdata_shape: {mlip_df.shape}")
    print(f"Collecting complete, starting phase diagram construction...")
    ehull_df, eform_df = construct_phase_diagrams(
        hdp_df=hdp_df,
        subsys_MLIPenergy_df=mlip_df,
        dataframe_savedir= Path("../AnalysisResults"),
        phasediagram_savedir= os.path.join(data_dir, "PhaseDiagramData"),
    )
    print(f"Ehull calculation complete, simplified report from {data_dir}:")

    summarize_results(mlip_df, ehull_df)

['GRACE-3L-OMAT-large-ft-AM', 'SevenNet-MPALOE', 'SevenNet-omat24', 'PET-OAM-XL']
collecting data from ExpensiveMLIPs
mlipdata_shape: (12154, 8)
Ehull calculation complete, simplified report from ExpensiveMLIPs:
MLIP: 	 NaN vals:
E_GRACE-3L-OMAT-large-ft-AM 	 258
E_SevenNet-MPALOE 	 3
E_SevenNet-omat24 	 2
E_PET-OAM-XL 	 2
MLIP: 	 Missing HDPs: 	 share stable (<=100meV/atom):
GRACE-3L-OMAT-large-ft-AM 	 258 	 0.7237302977232924
SevenNet-MPALOE 	 259 	 0.7459483136224266
SevenNet-omat24 	 258 	 0.6957092819614711
PET-OAM-XL 	 258 	 0.7092819614711033
MLIP: 	 Missing HDPs: 	 share stable (<=150meV/atom):
GRACE-3L-OMAT-large-ft-AM 	 258 	 0.8489492119089317
SevenNet-MPALOE 	 259 	 0.8576434515987735
SevenNet-omat24 	 258 	 0.8239929947460596
PET-OAM-XL 	 258 	 0.8226795096322241
MLIP: 	 Missing HDPs: 	 share stable (<=200meV/atom):
GRACE-3L-OMAT-large-ft-AM 	 258 	 0.9001751313485113
SevenNet-MPALOE 	 259 	 0.9093298291721419
SevenNet-omat24 	 258 	 0.8905429071803853
PET-OAM-XL 	 258 	 0

In [ ]:
for col in mlip_df.columns:
    if col.startswith("E_"):
        miss_data = mlip_df.loc[mlip_df[col].isna()]
        miss_data[["chemsys", "nsites", "structure_dict"]].to_csv(
            f"MissingStrucs_{col.removeprefix('E_')}.csv"
        )

In [ ]:
allowed_els = [
    "Ac",
    "Ag",
    "Al",
    "Ar",
    "As",
    "Au",
    "B",
    "Ba",
    "Be",
    "Bi",
    "Br",
    "C",
    "Ca",
    "Cd",
    "Ce",
    "Cl",
    "Co",
    "Cr",
    "Cs",
    "Cu",
    "Dy",
    "Er",
    "Eu",
    "F",
    "Fe",
    "Ga",
    "Gd",
    "Ge",
    "H",
    "He",
    "Hf",
    "Hg",
    "Ho",
    "I",
    "In",
    "Ir",
    "K",
    "Kr",
    "La",
    "Li",
    "Lu",
    "Mg",
    "Mn",
    "Mo",
    "N",
    "Na",
    "Nb",
    "Nd",
    "Ne",
    "Ni",
    "Np",
    "O",
    "Os",
    "P",
    "Pa",
    "Pb",
    "Pd",
    "Pm",
    "Pr",
    "Pt",
    "Pu",
    "Rb",
    "Re",
    "Rh",
    "Ru",
    "S",
    "Sb",
    "Sc",
    "Se",
    "Si",
    "Sm",
    "Sn",
    "Sr",
    "Ta",
    "Tb",
    "Tc",
    "Te",
    "Th",
    "Ti",
    "Tl",
    "Tm",
    "U",
    "V",
    "W",
    "Xe",
    "Y",
    "Yb",
    "Zn",
    "Zr",
]
drop_index = mlip_df.apply(
    lambda row: False in [x in allowed_els for x in row["chemsys"].split("-")], axis=1
)
mlip_df.drop(drop_index[drop_index == True].index)

,structure_dict,chemsys,nsites,structure,E_SevenNet-MPALOE,E_SevenNet-omat24,E_PET-OAM-XL
mp-1225895,"{'@module': 'pymatgen.core.structure', '@class...",Cs-Rb,2,"[[0. 0. 0.] Cs, [3.03786378 0. 4.19497...",-39.01326,-1.790802,-1.804725
mp-1185559,"{'@module': 'pymatgen.core.structure', '@class...",Cs-Rb,4,"[[0. 0. 0.] Cs, [2.92420802 2.92420802 2.92420...",-66.810379,-3.631001,-3.744776
mp-1183945,"{'@module': 'pymatgen.core.structure', '@class...",Cs-Rb,4,"[[0. 0. 0.] Cs, [0. 3.69598298 3.69598...",-66.85466,-3.674257,-3.716936
mp-1184016,"{'@module': 'pymatgen.core.structure', '@class...",Cs-Rb,4,"[[0. 0. 0.] Cs, [4.44089210e-16 3.70126506e+00...",-66.857254,-3.677019,-3.744331
mp-862689,"{'@module': 'pymatgen.core.structure', '@class...",Cs-Rb,8,[[-6.10753338e-06 3.19002770e+00 2.22351732e...,-178.404694,-6.982025,-7.048181
...,...,...,...,...,...,...,...
3749_CsCsIrBr,"{'@module': 'pymatgen.core.structure', '@class...",Br-Cs-Ir,10,"[[0. 0. 0.] Cs, [5.73456991 5.73456991 5.73456...",-220.834442,-33.010117,-32.988205
3750_CsCsAuF,"{'@module': 'pymatgen.core.structure', '@class...",Au-Cs-F,10,"[[0. 0. 0.] Cs, [4.83390697 4.83390697 4.83390...",-166.809906,-37.208961,-37.719357
3751_CsCsAuCl,"{'@module': 'pymatgen.core.structure', '@class...",Au-Cl-Cs,10,"[[0. 0. 0.] Cs, [5.5867253 5.5867253 5.5867253...",-176.654877,-29.173389,-29.201574
3752_CsCsTlF,"{'@module': 'pymatgen.core.structure', '@class...",Cs-F-Tl,10,"[[0. 0. 0.] Cs, [4.90913393 4.90913393 4.90913...",-173.210083,-39.885536,-40.546158


In [ ]:
ehull_df = pd.read_csv(
    "../AnalysisResults/HDP_Ehull_data_full.csv",
    index_col=0,
)
eform_df = pd.read_csv(
    "../AnalysisResults/HDP_Eform_data_full.csv",
    index_col=0,
)

ehull_avg = ehull_df.mean(axis=1)
ehull_avg.name = "Ehull_avg"
eform_avg = eform_df.mean(axis=1)
eform_avg.name = "Eform_avg"

ehull_std = ehull_df.std(axis=1)
ehull_std.name = "Ehull_std"
eform_std = eform_df.std(axis=1)
eform_std.name = "Eform_std"

In [8]:
ehull_df.dropna(how='all').shape

(2286, 4)

In [ ]:
def stable_counts(
    ehull_df: pd.DataFrame, cutoff_values: list[float] = [100, 150, 200]
) -> pd.DataFrame:
    counts_df = pd.DataFrame(index=ehull_df.index)
    counts_df["MLIP_entries"] = ehull_df.count(axis=1)
    for cutoff in cutoff_values:
        val = cutoff / 1000
        counts_df[f"Ehull <= {cutoff}meV"] = ehull_df.where(ehull_df < val).count(
            axis=1
        )

    return counts_df

In [ ]:
counts = stable_counts(ehull_df)
stab_data = pd.concat([eform_avg, eform_std, ehull_avg, ehull_std, counts], axis=1)
stab_data.to_csv("../AnalysisResults/HDP_Ehull_overview.csv")

In [ ]:
mlip_list = [
    "GRACE-3L-OMAT-large-ft-AM",
    "SevenNet-MPALOE",
    "SevenNet-omat24",
    "PET-OAM-XL",
]  # ,   ['Equiformer-v3']
missing_list = []
comps = {}
for mlip in mlip_list:
    misidx = pd.read_csv(f"MissingStrucs_{mlip}.csv", index_col=0)
    miss_ser = pd.Series(
        data={idx: np.int64(1) for idx in misidx.index if "_Cs" not in idx}, name=mlip
    )
    comps.update(
        {
            idx: Structure.from_dict(
                eval(misidx.loc[idx]["structure_dict"])
            ).composition.reduced_formula
            for idx in misidx.index
            if "_Cs" not in idx
        }
    )
    missing_list.append(miss_ser)
comp_series = pd.Series(comps, name="composition")
missing_list.append(comp_series)
missing_df = pd.DataFrame(missing_list).fillna(np.int64(0)).T
missing_df.to_csv("HDP_Ehull_MissingStrucs.csv")
missing_df

,GRACE-3L-OMAT-large-ft-AM,SevenNet-MPALOE,SevenNet-omat24,PET-OAM-XL,composition
mp-1214068,1.0,1.0,0,0,Cr3CdF6
mp-1212233,1.0,0,1.0,0,Mn3PtF6
mp-1207330,0,1.0,0,1.0,Hg3PtF6
mp-1212235,0,1.0,0,0,PdPt3F6
mp-1211674,0,0,1.0,0,Li3PF6
mp-1205972,0,0,0,1.0,MnZn3F6


In [ ]:
def comparative_histograms(
    hdp_df: pd.DataFrame,
    base_indices: list | pd.Index | pd.Series,
    compare_indices: list | pd.Index | pd.Series,
    base_label: str = "full set",
    compare_label: str = "unstable comps",
    **kwargs
):
    import plotly.graph_objects as go
    from pymatgen.core import Element

    col = hdp_df.columns.to_list()[0]
    base_count_b1 = hdp_df.loc[base_indices].groupby("element.B1").count()[col]
    base_count_b2 = hdp_df.loc[base_indices].groupby("element.B2").count()[col]

    elem_list = list(set(base_count_b2.index.to_list() + base_count_b1.index.to_list()))
    elem_map = {elem : Element(elem).Z for elem in elem_list if elem != 'Vac'}
    elem_map.update({'Vac':1})

    elem_idx = pd.Index(
        dict(sorted(elem_map.items() , key= lambda item: item[1])).keys()
    )
    print(elem_idx)

    base_count_ser = base_count_b1.reindex(elem_idx).fillna(0) + base_count_b2.reindex(
        elem_idx
    ).fillna(0)

    compare_count_b1 = hdp_df.loc[compare_indices].groupby("element.B1").count()[col]
    compare_count_b2 = hdp_df.loc[compare_indices].groupby("element.B2").count()[col]
    compare_count_ser = compare_count_b1.reindex(elem_idx).fillna(0) + compare_count_b2.reindex(elem_idx).fillna(0)

    counter_df = pd.DataFrame({base_label: base_count_ser, compare_label: compare_count_ser})
    fig = go.Figure(data = [
        go.Bar(x=elem_idx, y = counter_df[base_label]/np.sum(counter_df[base_label]), name=base_label, marker_color="#575554"),
        go.Bar(x=elem_idx, y = counter_df[compare_label]/np.sum(counter_df[compare_label]), name=compare_label, marker_color="#E55107"),
    ])
    fig.update_traces(opacity=0.75)
    fig.show()

In [88]:
comparative_histograms(hdp_df=hdp_df, base_indices=hdp_df.index, compare_indices=plot_df[plot_df['Ehull_avg']> 0.2].index)

Index(['Vac', 'Li', 'Be', 'B', 'N', 'Na', 'Mg', 'Al', 'Si', 'P', 'K', 'Ca',
       'Sc', 'Ti', 'V', 'Cr', 'Mn', 'Fe', 'Co', 'Ni', 'Cu', 'Zn', 'Ga', 'Ge',
       'As', 'Se', 'Rb', 'Sr', 'Y', 'Zr', 'Nb', 'Mo', 'Tc', 'Ru', 'Rh', 'Pd',
       'Ag', 'Cd', 'In', 'Sn', 'Sb', 'Te', 'Cs', 'Ba', 'La', 'Ce', 'Pr', 'Nd',
       'Pm', 'Sm', 'Eu', 'Gd', 'Tb', 'Dy', 'Ho', 'Er', 'Tm', 'Yb', 'Lu', 'Hf',
       'Ta', 'W', 'Re', 'Os', 'Ir', 'Pt', 'Au', 'Hg', 'Tl', 'Pb', 'Bi', 'Po',
       'Fr', 'Ra', 'Ac', 'Th', 'Pa', 'U', 'Np', 'Pu', 'Am', 'Cm'],
      dtype='str')
     full set  unstable comps
Vac     132.0             1.0
Li      109.0             3.0
Be       48.0            35.0
B        22.0            18.0
N        19.0            12.0
Na      154.0             3.0
Mg       68.0             7.0
Al       31.0             1.0
Si        1.0             0.0
P        27.0             8.0


Gtk-Message: 21:34:36.216: Not loading module "atk-bridge": The functionality is provided by GTK natively. Please try to not load it.
Gtk-Message: 21:34:36.276: Failed to load module "canberra-gtk-module"
Gtk-Message: 21:34:36.277: Failed to load module "canberra-gtk-module"
